# 강의 03 · 실습 2 — 랭체인 기초 · (4) 고난도 I

## 1. 문제상황

- 구름월드 채팅 상담창에는 「안녕하세요!! 저기요 ㅠㅠ 주차비가 얼마예요?? 감사합니다~~」처럼 인사말, 감탄 부호, 이모티콘이 붙은 질문이 들어옵니다.
- 지금의 안내 체인은 질문을 그대로 템플릿에 넣으므로, 군더더기가 많은 질문에서는 주제 판정이 흔들리고 답변에 인사말이 섞여 나옵니다.
- 상담 화면에는 답변을 「[주제] 답변 한 줄」 형식으로 붙여 넣어야 하는데, 지금은 topic과 answer를 사람이 옮겨 적어 형식을 맞춥니다.
- 담당자는 체인에 질문을 넣기 전에 군더더기를 걷어 내고, 체인 결과를 화면 형식 문자열로 바로 받기를 원합니다.

## 2. 문제와 목표

- **문제**: 질문의 군더더기가 체인에 그대로 들어가 판정을 흔들고, 체인 결과를 화면 형식으로 옮기는 일을 사람이 합니다.
- **목표**
  - 템플릿 앞에 질문 다듬기 함수를, 모델 뒤에 화면 형식 함수를 부품으로 끼웁니다.
    - 두 함수: 질문 다듬기(입력 딕셔너리를 받아 군더더기를 걷어 낸 딕셔너리를 돌려줌), 화면 형식(구조화 객체를 받아 「[주제] 답변」 문자열을 돌려줌)
  - 잡음 섞인 질문을 넣으면 「[주제] 답변」 문자열이 바로 나오는 체인을 만듭니다.
- **목표 달성 여부의 판정 기준**:
  - 잡음 섞인 질문 두 개를 넣었을 때, 다듬기 함수가 돌려준 질문에 인사말·감탄 부호·물결표가 없고,
  - 최종 결과가 `[주차] …`·`[환불] …` 형식의 문자열이며, 답변에 인사말이 섞이지 않은 것을 실행 기록에서 확인합니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec03_ex02_s4_diagram.svg)

## 4. 단계별 요구사항

1. **출력 스키마와 모델, 템플릿을 선언합니다.**
    - `topic`(주제)과 `answer`(답변) 두 문자열 칸을 가지는 `FaqAnswer` 클래스를 `BaseModel`을 상속해 선언하고, `init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")`로 모델 부품 `llm`을 만들고, `ChatPromptTemplate.from_messages`로 시스템 메시지(「너는 시설 안내 담당자다. 아래 FAQ만 근거로 답한다.」와 `{faq}` 빈칸)와 사용자 메시지(`{question}` 빈칸)를 가지는 템플릿 `prompt`를 만듭니다.
    - FAQ 문자열 `FAQ_CONTEXT`의 값은 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다.
2. **질문 다듬기 함수를 만듭니다.**
    - `clean_input(inputs: dict) -> dict`는 입력 딕셔너리의 `question` 값에서 인사말(「안녕하세요」「저기요」「감사합니다」), 물결표(`~`), 「ㅠㅠ」를 지우고, 감탄·물음 부호의 반복(`!!`·`???`)을 `?` 하나로 줄이고, 단어 뒤에 붙지 않은 `?`와 겹친 공백을 없앤 뒤, 같은 키의 딕셔너리로 돌려줍니다.
    - `faq` 값은 그대로 둡니다.
3. **화면 형식 함수를 만듭니다.**
    - `to_line(out: FaqAnswer) -> str`은 `f"[{out.topic}] {out.answer}"`를 돌려줍니다.
4. **부품을 조립합니다.**
    - `clean_input | prompt | llm.with_structured_output(FaqAnswer) | to_line`으로 체인을 만듭니다.
    - 파이썬 함수는 파이프 첫 위치에도 올 수 있습니다.
    - 첫 부품이 함수이면 `RunnableLambda`로 감싸 줍니다.
5. **실행합니다.**
    - 잡음 섞인 질문 두 개(「안녕하세요!! 저기요 ㅠㅠ 주차비가 얼마예요?? 감사합니다~~」, 「환불은요~~ 언제까지 되나요??? 감사합니다!!」)를 넣어, 다듬어진 질문과 최종 문자열을 함께 출력합니다.
    - 출력 줄의 이름은 「다듬기 시험」(다듬어진 질문)과 「결과」(최종 문자열)입니다.

## 5. 코드 골격 — LangChain 체인 3단

부품 선언, 조립, 실행의 세 단계입니다. 부품 선언에 파이썬 함수 두 개가 더해지고, 조립에서 함수를 템플릿 앞과 모델 뒤에 끼웁니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 부품 선언 | 모델·프롬프트·출력 파서를 각각 하나씩 선언합니다 | `class FaqAnswer`, `init_chat_model`, `ChatPromptTemplate`, `def clean_input`, `def to_line` | 1, 2, 3 |
| ② 조립 | 부품을 파이프 기호로 한 줄에 잇습니다 | `RunnableLambda(clean_input) | prompt | llm.with_structured_output(FaqAnswer) | to_line` | 4 |
| ③ 실행 | 입력을 넣어 체인을 돌리고, 필요하면 조각으로 받습니다 | `chain.invoke({"faq": ..., "question": ...})` | 5 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 API 키를 읽습니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

In [ ]:
import os

from dotenv import load_dotenv, find_dotenv
from pydantic import BaseModel

from langchain.chat_models import init_chat_model
import re

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")
print("준비를 마쳤습니다.")

# 주어진 자료
FAQ_CONTEXT = (
    "운영시간: 매일 09:30~21:00 / "
    "주차: 4,000대 규모, 최초 30분 무료 / "
    "환불: 이용일 전날까지 전액, 당일 50%"
)


### 단계 ① — 부품 선언 (요구사항 1, 2, 3)

- 스키마·모델·템플릿은 요구사항 1대로 선언합니다. FAQ 문자열은 단계 0에 주어져 있습니다.
- 질문 다듬기 함수는 딕셔너리를 받아 딕셔너리를 돌려줍니다. 템플릿이 딕셔너리를 기대하기 때문입니다. 화면 형식 함수는 `FaqAnswer` 객체를 받아 문자열을 돌려줍니다.

In [ ]:
# 여기에 단계 ①(스키마·모델·템플릿·함수 두 개 선언)을 작성합니다.

### 단계 ② — 조립 (요구사항 4)

- 파이프 첫 위치에 오는 파이썬 함수는 `RunnableLambda`로 감싸야 `|`가 동작합니다. 두 번째 위치부터는 함수를 그대로 이어도 됩니다.
- 부품 순서는 다듬기 → 템플릿 → 모델 → 화면 형식입니다. 앞 부품의 출력 모양이 뒤 부품의 입력 모양과 맞아야 합니다.

In [ ]:
# 여기에 단계 ②(함수를 앞뒤에 끼운 체인 조립)를 작성합니다.

### 단계 ③ — 실행 (요구사항 5)

- 입력은 `faq`·`question` 두 키를 가진 딕셔너리입니다. 다듬기는 체인 안에서 일어나므로 호출하는 쪽은 잡음을 신경 쓰지 않습니다.

In [ ]:
# 여기에 단계 ③(잡음 질문 두 개 실행)을 작성합니다.

## 7. 실행 결과 확인

셀을 위에서 아래로 모두 실행한 뒤 다음 세 가지를 확인합니다.

1. 단계 ①의 「다듬기 시험」 출력에 인사말, `!!`, `??`, `~`, `ㅠㅠ`가 없습니다.
2. 두 질문의 「결과」가 `[주차] …`·`[환불] …` 형식의 한 줄 문자열입니다. `topic`·`answer`를 옮겨 적는 코드가 없습니다.
3. 결과의 답변에 인사말이 섞여 있지 않고 FAQ 문장에 근거합니다.

세 가지가 모두 확인되면 완성입니다.